# E-commerce Landing Page A/B Test — Analysis

## Pre-registration (written 2026-10-22, before any test results were computed)

**Question:** Does the new landing page change the conversion rate compared with the old page?

| Item | Decision |
|---|---|
| Primary metric | Conversion rate = converted users ÷ users, per group |
| Hypotheses | H0: p_treatment = p_control  ·  H1: p_treatment ≠ p_control |
| Test | Two-proportion z-test, **two-sided**, with a 95% CI for the difference (treatment − control) |
| Significance level | **alpha = 0.05** |
| Main analysis | `ab_dbt.int_ab_users_clean` — 286,690 users; every user appearing more than once in raw is excluded (they may have seen both pages) |
| Sensitivity check | `ab_dbt.int_ab_users_sensitivity` — 290,584 users; mismatched rows dropped, each user's first remaining event kept. Reported alongside the main result, stating whether the conclusion changes |
| Validity check first | Sample ratio mismatch (SRM): chi-square test against a 50/50 split. If p < 0.01, randomization is suspect and the result is not trusted |

**What had been seen before writing this:** row and user counts per group, and the *raw, pre-cleaning*
conversion rates from data profiling on 2026-10-21 (control 12.04%, treatment 11.89%). No conversion rates
or test statistics had been computed on either cleaned table.


## 6a. Pull the clean data

Both user tables come from the dbt models in BigQuery. Before going further, check they match the counts
dbt verified: one row per user, and the expected users per group. Only user counts are shown here;
conversion rates are first computed in 6c.

In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT = "ab-test-portfolio-512322"
client = bigquery.Client(project=PROJECT)


def load(table):
    sql = f"SELECT * FROM `{PROJECT}.ab_dbt.{table}`"
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


clean = load("int_ab_users_clean")
sens = load("int_ab_users_sensitivity")

# Sanity checks against the dbt-verified counts (users only, no conversion rates yet).
EXPECTED = {
    "clean": {"control": 143_293, "treatment": 143_397},
    "sensitivity": {"control": 145_274, "treatment": 145_310},
}
for name, df in [("clean", clean), ("sensitivity", sens)]:
    users = df["experiment_group"].value_counts().sort_index()
    assert df["user_id"].is_unique, f"{name}: duplicate user_id"
    assert users.to_dict() == EXPECTED[name], f"{name}: {users.to_dict()}"
    print(f"{name:<12} users={len(df):,}  " + "  ".join(f"{g}={n:,}" for g, n in users.items()))

clean        users=286,690  control=143,293  treatment=143,397
sensitivity  users=290,584  control=145,274  treatment=145,310


## 6b. Sample ratio mismatch (SRM) check

Users were meant to be split 50/50. A chi-square test compares the observed group sizes with a perfect
50/50 split. Per the pre-registration, p < 0.01 means randomization is suspect and the result is not trusted.

In [2]:
from scipy.stats import chisquare

SRM_ALPHA = 0.01


def srm_check(name, df):
    obs = df["experiment_group"].value_counts().sort_index()
    stat, p = chisquare(obs.values, [obs.sum() / 2] * 2)
    share = obs["treatment"] / obs.sum()
    verdict = "SRM detected" if p < SRM_ALPHA else "no SRM"
    print(
        f"{name:<12} control={obs['control']:,}  treatment={obs['treatment']:,}  "
        f"diff={obs['treatment'] - obs['control']:+,}  treatment share={share:.3%}  "
        f"chi2={stat:.4f}  p={p:.4f}  -> {verdict}"
    )


srm_check("clean", clean)
srm_check("sensitivity", sens)

clean        control=143,293  treatment=143,397  diff=+104  treatment share=50.018%  chi2=0.0377  p=0.8460  -> no SRM
sensitivity  control=145,274  treatment=145,310  diff=+36  treatment share=50.006%  chi2=0.0045  p=0.9468  -> no SRM


**SRM result: no sample ratio mismatch.** Treatment holds 50.018% of users in the main table
(+104 users, p = 0.846) and 50.006% in the sensitivity table (+36 users, p = 0.947), both far above the
pre-registered 0.01 threshold. A gap this small is what random assignment alone produces, so the split
looks healthy and the comparison in 6c can be trusted on this check. Excluding the 3,894 repeat users
did not unbalance the groups.